In [1]:
from pyspark.sql import SparkSession

drivers = [
    # ==================== MINIO / S3 ====================
    # Hadoop AWS integration - provides s3a:// filesystem support
        "/opt/spark/external-jars/minio/hadoop-aws-3.3.4.jar",
    # AWS Java SDK - low-level S3 API implementation
        "/opt/spark/external-jars/minio/aws-java-sdk-bundle-1.12.262.jar",
    # WildFly OpenSSL - for SSL/TLS connections (optional)
        "/opt/spark/external-jars/minio/wildfly-openssl-1.0.7.Final.jar",
    # ==================== ICEBERG ====================
    # Apache Iceberg
        "/opt/spark/external-jars/iceberg/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",    
]

spark = (SparkSession.builder
         .appName("Iceberg")
         .master("spark://spark-master:7077")
         .config("spark.jars", ",".join(drivers))
    # Inceberg конфигурация
         .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
         .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
         .config("spark.sql.catalog.iceberg.type", "hadoop")
         .config("spark.sql.catalog.iceberg.warehouse", "s3a://iceberg-warehouse/")
         .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
         .getOrCreate()
)

26/03/12 13:20:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
# Тест: создаем таблицу
spark.sql("""
    CREATE TABLE iceberg.ololo_test5 (
        id INT,
        name STRING,
        ts TIMESTAMP
    ) USING iceberg
""")


26/03/12 13:20:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[]

In [3]:
# Вставляем данные
spark.sql("""
    INSERT INTO iceberg.ololo_test5 VALUES 
    (1, 'Alice', current_timestamp()),
    (2, 'Bob', current_timestamp()),
    (3, 'Упячка упячка, пыщь пыщь', current_timestamp())
""")

26/03/12 13:21:26 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/12 13:21:26 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/03/12 13:21:28 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/03/12 13:21:28 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.9
26/03/12 13:21:28 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
                                                                                

DataFrame[]

In [4]:
# Читаем данные
spark.sql("SELECT * FROM iceberg.ololo_test5").show()

+---+--------------------+--------------------+
| id|                name|                  ts|
+---+--------------------+--------------------+
|  1|               Alice|2026-03-12 13:21:...|
|  2|                 Bob|2026-03-12 13:21:...|
|  3|Упячка упячка, пы...|2026-03-12 13:21:...|
+---+--------------------+--------------------+



In [ ]:
# Возможно сработает
spark.sql("SHOW DATABASES FROM iceberg").show()

In [ ]:
# Или
spark.sql("SHOW SCHEMAS IN iceberg").show()

In [ ]:
# Создаем namespace
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.default")

In [ ]:
spark.sql("SELECT * FROM iceberg.default.test_table_5").show()

In [ ]:
# Посмотрим, как Spark видит конфигурацию
print("Warehouse path:", spark.conf.get("spark.sql.catalog.iceberg.warehouse"))

In [ ]:
# Создаем другой namespace
spark.sql("CREATE NAMESPACE iceberg.test_namespace")

In [ ]:
# Вставляем данные
spark.sql("""
    INSERT INTO iceberg.ololo_test VALUES 
    (1, 'Alice', current_timestamp()),
    (2, 'Bob', current_timestamp())
""")

In [ ]:
# Читаем данные
spark.sql("SELECT * FROM iceberg.ololo_test").show()

In [ ]:
# Проверим, где хранятся метаданные
spark.sql("SHOW CURRENT NAMESPACE").show()

In [ ]:
# Посмотрим схему таблицы
spark.sql("DESCRIBE EXTENDED iceberg.test_table").show(truncate=False)

In [ ]:
# Проверим данные
spark.sql("SELECT * FROM iceberg.test_table").show()

In [ ]:
# Узнаем, сколько снапшотов у таблицы (фишка Iceberg!)
spark.sql("SELECT * FROM iceberg.test_table.snapshots").show(truncate=False)

In [ ]:
# Посмотреть все каталоги
spark.sql("SHOW CATALOGS").show()

In [ ]:
# Создаем базу данных 'default' в каталоге iceberg
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.default")

In [ ]:
# Посмотреть все базы данных в каталоге iceberg
spark.sql("SHOW NAMESPACES IN iceberg").show()

In [ ]:
# Посмотреть историю таблицы
spark.sql("SELECT * FROM iceberg.test_table.history").show(truncate=False)


In [ ]:
# Посмотреть файлы, из которых состоит таблица
spark.sql("SELECT * FROM iceberg.test_table.files").show(truncate=False)


In [ ]:

# Время-путешествие (time travel) — фишка Iceberg!
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType

# Узнаем snapshot_id из предыдущего запроса
snapshot_id = 1361148785820279752

# Читаем данные на момент конкретного снапшота
spark.sql(f"""
    SELECT * FROM iceberg.test_table VERSION AS OF {snapshot_id}
""").show()